In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
import joblib

# Загрузка
df = pd.read_csv('../loan_data.csv')
print(f"Загружено {len(df)} строк")
df.head()

In [ ]:
def clean_data(df):
    """Очистка от выбросов"""
    df_clean = df.copy()
    
    # Удаляем аномальный возраст
    df_clean = df_clean[(df_clean['person_age'] >= 18) & (df_clean['person_age'] <= 100)]
    
    # Удаляем аномальный стаж
    df_clean = df_clean[df_clean['person_emp_exp'] <= df_clean['person_age'] - 16]
    df_clean = df_clean[df_clean['person_emp_exp'] >= 0]
    
    # Удаляем нулевые ставки
    df_clean = df_clean[df_clean['loan_int_rate'] > 0]
    
    # Удаляем аномальный кредитный рейтинг
    df_clean = df_clean[df_clean['credit_score'] >= 300]
    
    print(f"После очистки: {len(df_clean)} строк (удалено {len(df) - len(df_clean)})")
    return df_clean

df_clean = clean_data(df)

In [ ]:
# Кодируем бинарный признак
df_clean['previous_defaults'] = df_clean['previous_loan_defaults_on_file'].map({'Yes': 1, 'No': 0})

# Определяем признаки
feature_cols = ['person_age', 'person_gender', 'person_education', 'person_income', 
                'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
                'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
                'credit_score', 'previous_defaults']

X = df_clean[feature_cols]
y = df_clean['loan_status']

print(f"Признаков: {len(feature_cols)}")
print(f"Обучающих примеров: {len(X)}")
print(f"Доля одобрений: {y.mean():.2%}")

In [ ]:
# Числовые признаки
numeric_features = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
                    'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
                    'credit_score']

# Категориальные признаки
categorical_features = ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent']

# Трансформеры
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Тестируем предобработку
X_sample = X.head()
X_processed = preprocessor.fit_transform(X_sample)
print(f"После предобработки: {X_processed.shape[1]} признаков")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}")
print(f"Test: {len(X_test)}")
print(f"Train approval rate: {y_train.mean():.2%}")
print(f"Test approval rate: {y_test.mean():.2%}")

In [ ]:
# Создаем pipeline
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Обучаем
lr_pipeline.fit(X_train, y_train)

# Предсказания
y_pred_lr = lr_pipeline.predict_proba(X_test)[:, 1]

# Метрики
lr_auc = roc_auc_score(y_test, y_pred_lr)
print(f"=== Logistic Regression ===\n")
print(f"ROC-AUC: {lr_auc:.4f}")

# Дополнительные метрики
y_pred_class = lr_pipeline.predict(X_test)
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_class, target_names=['Refused', 'Approved']))

In [ ]:
# Создаем pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

# Обучаем
rf_pipeline.fit(X_train, y_train)

# Предсказания
y_pred_rf = rf_pipeline.predict_proba(X_test)[:, 1]

# Метрики
rf_auc = roc_auc_score(y_test, y_pred_rf)
print(f"=== Random Forest ===\n")
print(f"ROC-AUC: {rf_auc:.4f}")

# Важность признаков
feature_names = (numeric_features + 
                 list(rf_pipeline.named_steps['preprocessor']
                      .named_transformers_['cat']
                      .get_feature_names_out(categorical_features)))

importances = rf_pipeline.named_steps['classifier'].feature_importances_
feat_imp = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False).head(10)

print("\n=== Топ-10 важных признаков ===\n")
print(feat_imp)

# Визуализация
plt.figure(figsize=(10, 6))
plt.barh(feat_imp['feature'], feat_imp['importance'])
plt.xlabel('Importance')
plt.title('Random Forest - Top 10 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Устанавливаем если нет
# !pip install xgboost

from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss'))
])

xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, y_pred_xgb)

print(f"=== XGBoost ===\n")
print(f"ROC-AUC: {xgb_auc:.4f}")

In [ ]:
# Собираем результаты
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'ROC-AUC': [lr_auc, rf_auc, xgb_auc]
})
results = results.sort_values('ROC-AUC', ascending=False)
print("=== Сравнение моделей ===\n")
print(results)

# ROC кривые
plt.figure(figsize=(10, 8))

for name, y_pred in [('Logistic Regression', y_pred_lr), 
                      ('Random Forest', y_pred_rf),
                      ('XGBoost', y_pred_xgb)]:
    fpr, tpr, _ = roc_curve(y_test, y_pred)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test, y_pred):.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Выбираем лучшую модель
best_model = None
best_auc = 0
best_name = ""

if lr_auc > best_auc:
    best_auc = lr_auc
    best_model = lr_pipeline
    best_name = "LogisticRegression"

if rf_auc > best_auc:
    best_auc = rf_auc
    best_model = rf_pipeline
    best_name = "RandomForest"

if xgb_auc > best_auc:
    best_auc = xgb_auc
    best_model = xgb_pipeline
    best_name = "XGBoost"

print(f"✅ Лучшая модель: {best_name}")
print(f"✅ ROC-AUC на тесте: {best_auc:.4f}")

# Кросс-валидация
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='roc_auc')
print(f"\nCross-validation ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
# Сохраняем модель
joblib.dump(best_model, '../models/pipeline.pkl')
print("✅ Модель сохранена в ../models/pipeline.pkl")

# Сохраняем список признаков
feature_config = {
    'features': feature_cols,
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'target': 'loan_status',
    'model_type': best_name,
    'auc_score': float(best_auc),
    'training_date': str(pd.Timestamp.now())
}

import json
with open('../config/features_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)
    
print("✅ Конфигурация сохранена в ../config/features_config.json")